# Gerador de arquivos de entrada para Checkpoint_6 | Dynamic Programming

Este notebook gera instâncias **reproduzíveis e diferentes por grupo** para as duas questões sobre recuperação financeira de varejistas por meio de precificação online.

Quatro arquivos produzidos por grupo:

- `q1_catalogo_<grupo>.csv`
- `q1_config_<grupo>.json`
- `q2_catalogo_<grupo>.csv`
- `q2_config_<grupo>.json`
- `manifesto_dados.json`

A `seed_base` é definida como a **soma do último dígito da matrícula de cada integrante**.  

> O notebook usa apenas bibliotecas comuns de Python: `numpy`, `pandas`, `json`, `hashlib` e `pathlib`.


## Bloco 01 — Importações e pasta de saída

In [ ]:
# Bloco 01 — Importações e pasta de saída

from pathlib import Path
import hashlib
import json
import math

import numpy as np
import pandas as pd

PASTA_SAIDA = Path("dados_gerados") # alterar para o path escolhido para o trabalho
PASTA_SAIDA.mkdir(parents=True, exist_ok=True) # criacao do diretorio

print(f"Pasta de saída: {PASTA_SAIDA.resolve()}")

Pasta de saída: /mnt/data/dados_gerados


## Bloco 02 — Identificação do grupo e matrículas
### Nenhum outro bloco do notebook deverá ser alterado. Modificações no gerador invalidam a instância produzida.

In [ ]:
# Bloco 02 — Unico Bloco para editar

# Identificador do Grupo (Nome)
GRUPO = "G01"

# Podem ser strings ou números. O código considera apenas o último algarismo.
# Exemplo -- colocar as matriculas na variavel | lista abaixo
MATRICULAS = [
    "RM123456",
    "RM234567",
    "RM345678",
]

print("Grupo:", GRUPO)
print("Quantidade de integrantes:", len(MATRICULAS))

Grupo: G01
Quantidade de integrantes: 3


## Blocos a não serem alterados

## Bloco 03 — Cálculo da seed do grupo

In [ ]:
# Bloco 03 — Cálculo automático da seed do grupo

def ultimo_digito(valor):
    digitos = [c for c in str(valor) if c.isdigit()]

    if not digitos:
        raise ValueError(
            f"A matrícula {valor!r} não possui algarismos."
        )

    return int(digitos[-1])

def numero_do_grupo(codigo_grupo):
    digitos = [c for c in str(codigo_grupo) if c.isdigit()]

    if not digitos:
        raise ValueError(
            "O código do grupo deve conter um número. Exemplo: G07."
        )

    return int("".join(digitos))

ultimos_digitos = [
    ultimo_digito(matricula)
    for matricula in MATRICULAS
]

seed_base = sum(ultimos_digitos)

numero_grupo = numero_do_grupo(GRUPO)

seed_final = numero_grupo * 1000 + seed_base


print("Últimos dígitos:", ultimos_digitos)

print("Número do grupo:", numero_grupo)

print("seed_base:", seed_base)

print("seed_final:", seed_final)

Últimos dígitos: [6, 7, 8]
seed_base: 21
offset: 0
seed_final: 21


## Bloco 04 — Funções auxiliares

In [ ]:
# Bloco 04 — Funções auxiliares

def arredondar_preco(valor):
    """
    Ajusta um valor monetário para terminar em ,90.

    Exemplo:
    153.27 -> 153.90
    """

    valor = max(float(valor), 1.0)

    parte_inteira = math.floor(valor)

    return round(parte_inteira + 0.90, 2)


def salvar_json(objeto, caminho):
    """
    Salva um dicionário Python em um arquivo JSON.

    O parâmetro ensure_ascii=False preserva corretamente
    caracteres acentuados da língua portuguesa.
    """

    caminho = Path(caminho)

    with caminho.open(
        "w",
        encoding="utf-8"
    ) as arquivo:

        json.dump(
            objeto,
            arquivo,
            ensure_ascii=False,
            indent=2
        )


def sha256_arquivo(caminho):
    """
    Calcula o hash SHA-256 de um arquivo. Protecao de identificacao.

    O hash será utilizado para verificar se os arquivos
    de entrada foram alterados depois de sua geração.
    """

    hash_sha256 = hashlib.sha256()

    with open(caminho, "rb") as arquivo:

        for bloco in iter(
            lambda: arquivo.read(65536),
            b""
        ):

            hash_sha256.update(bloco)

    return hash_sha256.hexdigest()


def rng_independente(seed, deslocamento):
    """
    Cria um gerador de números pseudoaleatórios independente.

    O deslocamento permite que diferentes partes da prova
    utilizem a mesma seed_final sem gerar exatamente a mesma
    sequência de números.

    Exemplo:

    Q1 -> seed_final + 101
    Q2 -> seed_final + 201
    """

    seed_local = int(seed) + int(deslocamento)

    return np.random.default_rng(seed_local)


print("Funções auxiliares carregadas com sucesso.")

## Bloco 05 — Parâmetros das categorias de varejo

In [ ]:
# Bloco 05 — Parâmetros das categorias de varejo

CATEGORIAS = {
    "Eletronicos": {
        "custo": (180, 1800), "markup": (1.18, 1.55),
        "estoque": (20, 160), "frete": (18, 70),
        "elasticidade": (1.0, 2.2)
    },
    "Eletrodomesticos": {
        "custo": (350, 2600), "markup": (1.16, 1.48),
        "estoque": (15, 100), "frete": (35, 160),
        "elasticidade": (0.8, 1.8)
    },
    "Informatica": {
        "custo": (120, 2200), "markup": (1.20, 1.60),
        "estoque": (20, 150), "frete": (15, 65),
        "elasticidade": (1.1, 2.3)
    },
    "Moveis": {
        "custo": (250, 2100), "markup": (1.20, 1.65),
        "estoque": (12, 90), "frete": (50, 220),
        "elasticidade": (0.7, 1.5)
    },
    "Moda": {
        "custo": (25, 220), "markup": (1.70, 2.80),
        "estoque": (60, 500), "frete": (8, 28),
        "elasticidade": (1.2, 2.6)
    },
    "Beleza": {
        "custo": (12, 180), "markup": (1.65, 2.70),
        "estoque": (80, 600), "frete": (6, 22),
        "elasticidade": (1.0, 2.4)
    },
    "Casa_Utilidades": {
        "custo": (18, 450), "markup": (1.45, 2.20),
        "estoque": (50, 360), "frete": (8, 45),
        "elasticidade": (0.9, 2.0)
    }
}

print("Categorias disponíveis:", ", ".join(CATEGORIAS.keys()))

Categorias disponíveis: Eletronicos, Eletrodomesticos, Informatica, Moveis, Moda, Beleza, Casa_Utilidades


## Bloco 06 — Questao 01 == Geracao de catalogo

In [ ]:
# Bloco 06 — Geração do catálogo da Questão 1

def gerar_q1_catalogo(seed, n_skus=30):
    rng = rng_independente(seed, 101)
    categorias = list(CATEGORIAS.keys())
    linhas = []

    for i in range(1, n_skus + 1):
        categoria = rng.choice(categorias)
        p = CATEGORIAS[categoria]

        custo = round(rng.uniform(*p["custo"]), 2)
        markup = rng.uniform(*p["markup"])
        preco = arredondar_preco(custo * markup)

        estoque = int(rng.integers(p["estoque"][0], p["estoque"][1] + 1))
        demanda = int(max(1, round(estoque * rng.uniform(0.45, 1.25))))

        taxa_marketplace = round(rng.uniform(0.08, 0.18), 4)
        custo_logistico = round(rng.uniform(*p["frete"]), 2)
        elasticidade = round(rng.uniform(*p["elasticidade"]), 3)
        idade_estoque = int(rng.integers(5, 361))

        linhas.append({
            "sku": f"Q1-{i:03d}",
            "categoria": categoria,
            "custo_unitario": custo,
            "preco_atual": preco,
            "estoque": estoque,
            "demanda_base_90d": demanda,
            "elasticidade": elasticidade,
            "taxa_marketplace": taxa_marketplace,
            "custo_logistico": custo_logistico,
            "idade_estoque_dias": idade_estoque,
        })

    return pd.DataFrame(linhas)

q1_catalogo = gerar_q1_catalogo(seed_final, n_skus=30)
q1_catalogo.head()

,sku,categoria,custo_unitario,preco_atual,estoque,demanda_base_90d,elasticidade,taxa_marketplace,custo_logistico,idade_estoque_dias
0,Q1-001,Casa_Utilidades,93.87,162.9,306,189,1.750,0.1623,26.70,132
1,Q1-002,Eletronicos,1648.99,2510.9,63,60,2.012,0.1434,25.35,16
2,Q1-003,Eletronicos,582.99,812.9,91,48,1.695,0.0883,49.98,325
3,Q1-004,Informatica,1992.72,2623.9,42,28,2.047,0.1630,16.73,302
4,Q1-005,Eletrodomesticos,1526.46,1944.9,88,65,1.128,0.1187,76.23,180


## Bloco 07 — Questão 1== Configuração financeira

In [ ]:
# Bloco 07 — Configuração financeira da Questão 1

def gerar_q1_config(seed, catalogo):
    rng = rng_independente(seed, 102)

    unidades_base = np.minimum(
        catalogo["estoque"].to_numpy(),
        catalogo["demanda_base_90d"].to_numpy()
    )

    margem_unitaria = (
        catalogo["preco_atual"] * (1 - catalogo["taxa_marketplace"])
        - catalogo["custo_unitario"]
        - catalogo["custo_logistico"]
    ).clip(lower=0)

    caixa_base_estimado = float(np.sum(unidades_base * margem_unitaria))
    receita_base_estimada = float(np.sum(unidades_base * catalogo["preco_atual"]))

    config = {
        "grupo": GRUPO,
        "seed_base": int(seed_base),
        "seed_final": int(seed),
        "horizonte_dias": 90,
        "meta_caixa_90d": round(caixa_base_estimado * rng.uniform(0.82, 0.96), 2),
        "divida_total_90d": round(caixa_base_estimado * rng.uniform(0.78, 0.92), 2),
        "orcamento_promocional": round(receita_base_estimada * rng.uniform(0.025, 0.055), 2),
        "margem_minima_percentual": round(rng.uniform(0.05, 0.10), 4),
        "niveis_preco_relativos": [0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10],
        "observacao": "Os níveis de preço são multiplicadores aplicados sobre o preco_atual."
    }
    return config

q1_config = gerar_q1_config(seed_final, q1_catalogo)
q1_config

{'grupo': 'G01',
 'seed_base': 21,
 'seed_final': 21,
 'horizonte_dias': 90,
 'meta_caixa_90d': 356131.05,
 'divida_total_90d': 306342.64,
 'orcamento_promocional': 83046.4,
 'margem_minima_percentual': 0.0592,
 'niveis_preco_relativos': [0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1],
 'observacao': 'Os níveis de preço são multiplicadores aplicados sobre o preco_atual.'}

## Bloco 08 — Questão 2 == Geração do catálogo

In [ ]:
# Bloco 08 — Geração do catálogo da Questão 2

def gerar_q2_catalogo(seed, n_skus=24):
    rng = rng_independente(seed, 201)
    categorias = list(CATEGORIAS.keys())
    linhas = []

    for i in range(1, n_skus + 1):
        categoria = rng.choice(categorias)
        p = CATEGORIAS[categoria]

        custo = round(rng.uniform(*p["custo"]), 2)
        markup = rng.uniform(*p["markup"])
        preco = arredondar_preco(custo * markup)

        estoque = int(rng.integers(p["estoque"][0], p["estoque"][1] + 1))
        demanda_semanal = int(max(1, round(estoque * rng.uniform(0.035, 0.16))))

        taxa_marketplace = round(rng.uniform(0.08, 0.18), 4)
        custo_logistico = round(rng.uniform(*p["frete"]), 2)
        elasticidade = round(rng.uniform(*p["elasticidade"]), 3)
        preco_concorrente = arredondar_preco(preco * rng.uniform(0.90, 1.10))

        linhas.append({
            "sku": f"Q2-{i:03d}",
            "categoria": categoria,
            "custo_unitario": custo,
            "preco_atual": preco,
            "estoque_inicial": estoque,
            "demanda_base_semanal": demanda_semanal,
            "elasticidade": elasticidade,
            "taxa_marketplace": taxa_marketplace,
            "custo_logistico_base": custo_logistico,
            "preco_concorrente_base": preco_concorrente,
        })

    return pd.DataFrame(linhas)

q2_catalogo = gerar_q2_catalogo(seed_final, n_skus=24)
q2_catalogo.head()

,sku,categoria,custo_unitario,preco_atual,estoque_inicial,demanda_base_semanal,elasticidade,taxa_marketplace,custo_logistico_base,preco_concorrente_base
0,Q2-001,Informatica,1583.58,2357.9,69,10,1.766,0.1696,19.00,2373.9
1,Q2-002,Moda,118.36,251.9,304,14,1.863,0.1168,9.88,256.9
2,Q2-003,Eletronicos,583.09,836.9,123,14,1.880,0.1541,43.08,805.9
3,Q2-004,Casa_Utilidades,343.66,746.9,127,8,1.274,0.1372,19.92,740.9
4,Q2-005,Eletronicos,1796.78,2741.9,94,10,1.839,0.0841,48.24,2522.9


## Bloco 09 — Questão 2 == Configuração financeira e estocástica

In [ ]:
# Bloco 09 — Configuração financeira e estocástica da Questão 2

def gerar_q2_config(seed, catalogo):
    rng = rng_independente(seed, 202)

    margem_unitaria = (
        catalogo["preco_atual"] * (1 - catalogo["taxa_marketplace"])
        - catalogo["custo_unitario"]
        - catalogo["custo_logistico_base"]
    ).clip(lower=0)

    unidades_semanais = np.minimum(
        catalogo["demanda_base_semanal"].to_numpy(),
        np.maximum(1, catalogo["estoque_inicial"].to_numpy() / 12)
    )

    caixa_semanal_base = float(np.sum(unidades_semanais * margem_unitaria))
    receita_semanal_base = float(np.sum(unidades_semanais * catalogo["preco_atual"]))
    caixa_12s_base = caixa_semanal_base * 12

    total_dividas = caixa_12s_base * rng.uniform(0.72, 0.88)
    pesos = np.array([0.24, 0.31, 0.45])

    config = {
        "grupo": GRUPO,
        "seed_base": int(seed_base),
        "seed_final": int(seed),
        "numero_semanas": 12,
        "caixa_inicial": round(caixa_12s_base * rng.uniform(0.04, 0.09), 2),
        "dividas": {
            "semana_4": round(total_dividas * pesos[0], 2),
            "semana_8": round(total_dividas * pesos[1], 2),
            "semana_12": round(total_dividas * pesos[2], 2),
        },
        "variabilidade": {
            "sigma_demanda": round(rng.uniform(0.08, 0.18), 4),
            "sigma_preco_concorrente": round(rng.uniform(0.03, 0.10), 4),
            "sigma_custo_logistico": round(rng.uniform(0.05, 0.15), 4),
        },
        "orcamento_promocional_semanal": round(
            receita_semanal_base * rng.uniform(0.025, 0.050), 2
        ),
        "niveis_preco_relativos": [0.82, 0.88, 0.94, 1.00, 1.06, 1.12],
        "n_simulacoes_recomendadas": 2000,
        "limite_risco_falha": 0.10,
        "observacao": (
            "As variabilidades são parâmetros de dispersão relativos. "
            "A modelagem probabilística deve ser explicada pelos alunos."
        )
    }
    return config

q2_config = gerar_q2_config(seed_final, q2_catalogo)
q2_config

{'grupo': 'G01',
 'seed_base': 21,
 'seed_final': 21,
 'numero_semanas': 12,
 'caixa_inicial': 20069.12,
 'dividas': {'semana_4': np.float64(68627.23),
  'semana_8': np.float64(88643.5),
  'semana_12': np.float64(128676.05)},
 'variabilidade': {'sigma_demanda': 0.1475,
  'sigma_preco_concorrente': 0.0518,
  'sigma_custo_logistico': 0.1199},
 'orcamento_promocional_semanal': 5677.83,
 'niveis_preco_relativos': [0.82, 0.88, 0.94, 1.0, 1.06, 1.12],
 'n_simulacoes_recomendadas': 2000,
 'limite_risco_falha': 0.1,
 'observacao': 'As variabilidades são parâmetros de dispersão relativos. A modelagem probabilística deve ser explicada pelos alunos.'}

## Bloco 10 — Validação das instâncias

In [ ]:
# Bloco 10 — Validação das instâncias

def validar_catalogos(q1, q2):
    erros = []

    if q1["sku"].duplicated().any():
        erros.append("Q1 possui SKU duplicado.")
    if q2["sku"].duplicated().any():
        erros.append("Q2 possui SKU duplicado.")

    if (q1["custo_unitario"] <= 0).any() or (q2["custo_unitario"] <= 0).any():
        erros.append("Há custo unitário não positivo.")

    if (q1["preco_atual"] <= 0).any() or (q2["preco_atual"] <= 0).any():
        erros.append("Há preço não positivo.")

    if (q1["estoque"] <= 0).any() or (q2["estoque_inicial"] <= 0).any():
        erros.append("Há estoque não positivo.")

    if not q1["taxa_marketplace"].between(0, 1).all():
        erros.append("Q1 possui taxa de marketplace fora de [0,1].")
    if not q2["taxa_marketplace"].between(0, 1).all():
        erros.append("Q2 possui taxa de marketplace fora de [0,1].")

    if erros:
        raise ValueError("\n".join(erros))

    return True

validar_catalogos(q1_catalogo, q2_catalogo)
print("Validação concluída: instâncias consistentes.")

Validação concluída: instâncias consistentes.


## Bloco 11 — Gravação dos quatro arquivos de entrada

In [1]:
# Bloco 11 — Gravação dos quatro arquivos de entrada -- atencao ao endereco

pasta_grupo = PASTA_SAIDA / GRUPO
pasta_grupo.mkdir(parents=True, exist_ok=True)

arquivo_q1_csv = pasta_grupo / f"q1_catalogo_{GRUPO}.csv"
arquivo_q1_json = pasta_grupo / f"q1_config_{GRUPO}.json"
arquivo_q2_csv = pasta_grupo / f"q2_catalogo_{GRUPO}.csv"
arquivo_q2_json = pasta_grupo / f"q2_config_{GRUPO}.json"

q1_catalogo.to_csv(arquivo_q1_csv, index=False, encoding="utf-8-sig")
salvar_json(q1_config, arquivo_q1_json)

q2_catalogo.to_csv(arquivo_q2_csv, index=False, encoding="utf-8-sig")
salvar_json(q2_config, arquivo_q2_json)

print("Arquivos gerados:")
for p in [arquivo_q1_csv, arquivo_q1_json, arquivo_q2_csv, arquivo_q2_json]:
    print(" -", p)

NameError: name 'PASTA_SAIDA' is not defined

## Bloco 12 — Manifesto e hashes SHA-256

In [ ]:
# Bloco 12 — Manifesto, hashes e identificação da instância - protecao de arquivo

hashes = {
    arquivo_q1_csv.name:
        sha256_arquivo(arquivo_q1_csv),

    arquivo_q1_json.name:
        sha256_arquivo(arquivo_q1_json),

    arquivo_q2_csv.name:
        sha256_arquivo(arquivo_q2_csv),

    arquivo_q2_json.name:
        sha256_arquivo(arquivo_q2_json),
}


hash_conjunto = hashlib.sha256(
    "".join(
        hashes[k]
        for k in sorted(hashes)
    ).encode("utf-8")
).hexdigest()


id_instancia = hash_conjunto[:12]


manifesto = {

    "grupo": GRUPO,

    "quantidade_integrantes":
        len(MATRICULAS),

    "seed_base":
        int(seed_base),

    "seed_final":
        int(seed_final),

    "id_instancia":
        id_instancia,

    "arquivos":
        hashes,

    "sha256_conjunto":
        hash_conjunto,

    "aviso":
        (
            "Os arquivos originais devem ser preservados. "
            "Qualquer alteração modificará o hash."
        )
}


arquivo_manifesto = (
    pasta_grupo /
    "manifesto_dados.json"
)


salvar_json(
    manifesto,
    arquivo_manifesto
)


print("Grupo:", GRUPO)

print("Seed base:", seed_base)

print("Seed final:", seed_final)

print("ID da instância:", id_instancia)

print()

print(
    json.dumps(
        manifesto,
        ensure_ascii=False,
        indent=2
    )
)

## Bloco 13 — Resumo visual da instância

In [ ]:
# Bloco 13 — Resumo visual da instância

resumo = pd.DataFrame([
    {
        "Questao": "Q1",
        "SKUs": len(q1_catalogo),
        "Categorias": q1_catalogo["categoria"].nunique(),
        "Estoque_total": int(q1_catalogo["estoque"].sum()),
        "Meta_caixa": q1_config["meta_caixa_90d"],
        "Divida": q1_config["divida_total_90d"],
    },
    {
        "Questao": "Q2",
        "SKUs": len(q2_catalogo),
        "Categorias": q2_catalogo["categoria"].nunique(),
        "Estoque_total": int(q2_catalogo["estoque_inicial"].sum()),
        "Meta_caixa": np.nan,
        "Divida": round(sum(q2_config["dividas"].values()), 2),
    }
])

display(resumo)
print("\nPasta final:", pasta_grupo.resolve())

,Questao,SKUs,Categorias,Estoque_total,Meta_caixa,Divida
0,Q1,30,7,3676,356131.05,306342.64
1,Q2,24,7,3204,NaN,285946.78



Pasta final: /mnt/data/dados_gerados/G01
